In [9]:
import polars as pl

# load db_type 
db_type = pl.read_csv('ssdna_annotations/db_type_data.tsv', separator='\t', has_header=False)
db_type_new = (
    db_type
        .filter(pl.col('column_4') == 'RGBA(108, 254, 10, 100)')
        .with_columns([pl.col('column_1').str.replace_all("'", '').alias('column_1')])
)

In [15]:
from Bio import Phylo

def pd_for_subset(tree, subset_tips, treat_none_as=0.0, verbose=True):
    """
    Compute Faith's PD for the intersection of subset_tips and tips in the tree.
    - subset_tips: iterable of tip names (may include some not in the tree)
    - treat_none_as: numeric value to substitute for missing branch_length (default 0.0)
    - verbose: if True, prints a warning about missing tips
    Returns: pd_subset, pd_total, fraction
    """
    # map clade -> parent for upward traversal
    parent = {}
    for clade in tree.find_clades(order='preorder'):
        for child in clade.clades:
            parent[child] = clade

    # find tips in the tree
    name_to_clade = {c.name: c for c in tree.get_terminals() if c.name is not None}
    present_tips = [n for n in subset_tips if n in name_to_clade]
    missing_tips = [n for n in subset_tips if n not in name_to_clade]

    if verbose and missing_tips:
        print(f"Warning: {len(missing_tips)} tips not found in tree and will be ignored: {missing_tips}")

    if not present_tips:
        return 0.0, 0.0, float('nan')  # no overlap, PD = 0

    # collect unique edges (child clades)
    edges = set()
    for tip_name in present_tips:
        node = name_to_clade[tip_name]
        while node in parent:   # walk up to root
            edges.add(node)
            node = parent[node]

    def edge_length(child_clade):
        return child_clade.branch_length if child_clade.branch_length is not None else treat_none_as

    pd_subset = sum(edge_length(c) for c in edges)

    # total PD = sum of all branch lengths
    all_edges = set(parent.keys())
    pd_total = sum(edge_length(c) for c in all_edges)

    fraction = pd_subset / pd_total if pd_total > 0 else float('nan')
    return pd_subset, pd_total, fraction

# Example usage:
t = Phylo.read("ssdna_allvotureps_famgt20.nwk", "newick")
pd_sub, pd_all, frac = pd_for_subset(t, db_type_new['column_1'].to_list())
print("PD(subset) =", pd_sub, "PD(total) =", pd_all, "fraction =", frac)


PD(subset) = 796.355829887152 PD(total) = 3498.979033077152 fraction = 0.22759662814749781
